In [1]:
import sqlite3 # create local database
import os # for file path task


# This is the 'business_logic_2' notebook:
It handles the database for saving paste rules and the logic for monitoring the clipboard and simulating keystrokes.

In [3]:
import sqlite3 # import database module
import os # import os module
import time # import time module

KEYBOARD_AVAILABLE = False # set default flag
WIN32_AVAILABLE = False # set default flag

try:
    import keyboard # import keyboard library
    KEYBOARD_AVAILABLE = True # set flag to true
except ImportError:
    pass # handle missing library

try:
    import win32com.client # import word automation
    import win32clipboard # import clipboard module
    WIN32_AVAILABLE = True # set flag to true
except ImportError:
    pass # handle missing library

class BusinessLogic2: # define the main class for goal 2
    def __init__(self): # define new method
        self.db_path = "clipboard_history.db" # set database file name
        self._setup_database() # call setup method
        
    def _setup_database(self): # define method to create database
        conn = sqlite3.connect(self.db_path) # connect to database file
        cursor = conn.cursor() # create cursor
        cursor.execute('''CREATE TABLE IF NOT EXISTS clipboard_history 
                          (id INTEGER PRIMARY KEY AUTOINCREMENT, 
                           text_content TEXT, 
                           timestamp DATETIME DEFAULT CURRENT_TIMESTAMP)''') # create table if not exist
        conn.commit() # save changes
        conn.close() # close connection

    def get_clipboard_text(self): # define method to read clipboard
        if not WIN32_AVAILABLE: # check if library available
            return "Error: pywin32 is not installed." # return error
        try:
            win32clipboard.OpenClipboard() # open clipboard
            try:
                data = win32clipboard.GetClipboardData(win32clipboard.CF_UNICODETEXT) # read text
            except TypeError:
                data = "" # set empty if no text
            finally:
                win32clipboard.CloseClipboard() # close clipboard
            return data # return the text
        except Exception as e:
            return f"Error reading clipboard: {e}" # return error

    def paste_to_word_background(self): # define method to paste to word
        if not WIN32_AVAILABLE: # check if library available
            return False, "pywin32 is not installed." # return error
        try:
            text = self.get_clipboard_text() # get text from clipboard
            if not text or text.startswith("Error"): # check if text is valid
                return False, "No text in clipboard to paste." # return error
            try:
                word = win32com.client.GetActiveObject("Word.Application") # connect to open word
            except:
                word = win32com.client.Dispatch("Word.Application") # start word if not open
                word.Visible = False # keep word hidden
            selection = word.Selection # get cursor position
            selection.TypeText(text) # type the text
            selection.TypeParagraph() # press enter
            return True, "Pasted to Word in background successfully!" # return success
        except Exception as e:
            return False, f"Word Error: {str(e)}" # return error

    def save_to_database(self, text): # define method to save text
        if not text or text.startswith("Error"): # check if text is valid
            return False, "No valid text to save." # return error
        conn = sqlite3.connect(self.db_path) # connect to database
        cursor = conn.cursor() # create cursor
        cursor.execute("INSERT INTO clipboard_history (text_content) VALUES (?)", (text,)) # insert text
        conn.commit() # save changes
        conn.close() # close connection
        return True, "Text saved to database successfully!" # return success

    def get_all_history(self): # define method to get all history
        conn = sqlite3.connect(self.db_path) # connect to database
        cursor = conn.cursor() # create cursor
        cursor.execute("SELECT id, text_content, timestamp FROM clipboard_history ORDER BY timestamp DESC") # get all items sorted by newest
        rows = cursor.fetchall() # fetch all results
        conn.close() # close connection
        return rows # return the list

    def delete_history_item(self, item_id): # define method to delete item
        conn = sqlite3.connect(self.db_path) # connect to database
        cursor = conn.cursor() # create cursor
        cursor.execute("DELETE FROM clipboard_history WHERE id=?", (item_id,)) # delete item by id
        conn.commit() # save changes
        conn.close() # close connection
        return True # return success

    def paste_history_to_word(self, text): # define method to paste history to word
        if not WIN32_AVAILABLE: # check if library available
            return False, "pywin32 is not installed." # return error
        try:
            try:
                word = win32com.client.GetActiveObject("Word.Application") # connect to open word
            except:
                word = win32com.client.Dispatch("Word.Application") # start word if not open
                word.Visible = False # keep word hidden
            selection = word.Selection # get cursor position
            selection.TypeText(text) # type the text
            selection.TypeParagraph() # press enter
            return True, "History item pasted to Word!" # return success
        except Exception as e:
            return False, f"Error: {str(e)}" # return error

    def quick_copy(self): # define method for quick copy
        if not KEYBOARD_AVAILABLE: # check if library available
            return False, "keyboard library not installed." # return error
        keyboard.send('ctrl+c') # simulate ctrl+c
        time.sleep(0.2) # wait for clipboard
        return True, "Copy command sent!" # return success

    def quick_paste(self): # define method for quick paste
        if not KEYBOARD_AVAILABLE: # check if library available
            return False, "keyboard library not installed." # return error
        keyboard.send('ctrl+v') # simulate ctrl+v
        time.sleep(0.2) # wait for paste
        return True, "Paste command sent!" # return success

print("SUCCESS: BusinessLogic2 fully loaded!") # print successfully

SUCCESS: BusinessLogic2 fully loaded!
